In [1]:
import os
from dotenv import load_dotenv

from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import CodeInterpreterTool
from azure.identity import DefaultAzureCredential
from typing import Any
from pathlib import Path
from datetime import datetime


In [2]:
load_dotenv()
project_endpoint = os.environ["PROJECT_ENDPOINT"]

project_client = AIProjectClient(
    endpoint=project_endpoint,
    # 使用 Azure 默认凭据进行身份验证
    credential=DefaultAzureCredential(),
)

In [ ]:
from IPython.display import display, HTML, Image
from pathlib import Path


async def run_agent_with_visualization():
    html_output = "<h2>Azure AI Agent 执行</h2>"

    with project_client:
        # 创建 CodeInterpreterTool 实例
        code_interpreter = CodeInterpreterTool()

        # CodeInterpreterTool 需要包含在代理创建中
        # 确保为您的用例设置在 Azure AI Foundry 中部署的正确模型名称
        agent = project_client.agents.create_agent(
            model="gpt-4o",
            name="my-agent",
            instructions="你是一个有帮助的代理",
            tools=code_interpreter.definitions,
        )

  
        html_output += f"<div><strong>创建的代理</strong> ID: {agent.id}</div>"

        # 创建线程
        thread = project_client.agents.threads.create()
        html_output += f"<div><strong>创建的线程</strong> ID: {thread.id}</div>"

        # 用户查询 - 美化显示
        user_query = "请使用以下数据创建一个运营利润的条形图并提供文件给我：巴厘岛：100 名游客，巴黎：356 名游客，伦敦：900 名游客，东京：850 名游客"
        html_output += "<div style='margin:15px 0; padding:10px; background-color:#f5f5f5; border-left:4px solid #007bff; border-radius:4px;'>"
        html_output += "<strong>用户:</strong><br>"
        html_output += f"<div style='margin-left:15px'>{user_query}</div>"
        html_output += "</div>"

        # 创建消息
        message = project_client.agents.messages.create(
            thread_id=thread.id,
            role="user",
            content=user_query,
        )

        # 运行代理 - 显示"处理中"消息
        display(HTML(
            html_output + "<div style='color:#007bff'><i>处理请求中...</i></div>"))

        # 执行运行
        run = project_client.agents.runs.create_and_process(
            thread_id=thread.id, agent_id=agent.id)

        # 更新状态
        status_color = 'green' if run.status == 'completed' else 'red'
        html_output += f"<div><strong>运行完成</strong> 状态: <span style='color:{status_color}'>{run.status}</span></div>"

        if run.status == "failed":
            html_output += f"<div style='color:red'><strong>运行失败:</strong> {run.last_error}</div>"

        # 从线程获取消息
        messages = project_client.agents.messages.list(thread_id=thread.id)

        # 格式化助手响应
        html_output += "<div style='margin:15px 0; padding:10px; background-color:#f0f7ff; border-left:4px solid #28a745; border-radius:4px;'>"
        html_output += "<strong>助手:</strong><br>"

        # 根据实际结构处理消息
        # 首先，尝试获取助手的文本响应
        try:
            # 第一种方法 - 如果 messages 是带有 role 属性的对象列表
            assistant_msgs = [msg for msg in messages if hasattr(
                msg, 'role') and msg.role == "assistant"]

            if assistant_msgs:
                last_msg = assistant_msgs[-1]
                if hasattr(last_msg, 'content'):
                    if isinstance(last_msg.content, list):
                        for content_item in last_msg.content:
                            if hasattr(content_item, 'type') and content_item.type == "text":
                                html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{content_item.text.value}</div>"
                    elif isinstance(last_msg.content, str):
                        html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{last_msg.content}</div>"

            # 如果以上方法未找到消息，尝试不同的结构
            if not assistant_msgs:
                # 如果 messages 是带有属性的类
                if hasattr(messages, 'data'):
                    for msg in messages.data:
                        if hasattr(msg, 'role') and msg.role == "assistant":
                            if hasattr(msg, 'content'):
                                html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{msg.content}</div>"

        except Exception as e:
            html_output += f"<div style='color:red'><strong>处理消息时出错:</strong> {str(e)}</div>"

        html_output += "</div>"

        # 根据实际结构处理图像内容
        saved_images = []
        try:
            # 尝试将 image_contents 作为属性访问
            if hasattr(messages, 'image_contents'):
                for image_content in messages.image_contents:
                    file_id = image_content.image_file.file_id
                    file_name = f"{file_id}_image_file.png"
                    project_client.agents.save_file(
                        file_id=file_id, file_name=file_name)
                    saved_images.append(file_name)
                    html_output += f"<div style='margin-top:10px'><strong>生成的图像:</strong> {file_name}</div>"
        except Exception as e:
            html_output += f"<div style='color:orange'><i>注意: 未找到图像或处理图像时出错</i></div>"

        # 根据实际结构处理文件路径注释
        try:
            # 尝试将 file_path_annotations 作为属性访问
            if hasattr(messages, 'file_path_annotations'):
                for file_path_annotation in messages.file_path_annotations:
                    file_name = Path(file_path_annotation.text).name
                    project_client.agents.save_file(
                        file_id=file_path_annotation.file_path.file_id, file_name=file_name)
                    html_output += "<div style='margin:10px 0; padding:8px; background-color:#f8f9fa; border:1px solid #ddd; border-radius:4px;'>"
                    html_output += f"<strong>生成的文件:</strong> {file_name}<br>"
                    html_output += f"<strong>类型:</strong> {file_path_annotation.type}<br>"
                    html_output += "</div>"
        except Exception as e:
            html_output += f"<div style='color:orange'><i>注意: 未找到文件注释或处理文件时出错</i></div>"

        # 完成后删除代理
        project_client.agents.delete_agent(agent.id)
        html_output += "<div style='margin-top:10px'><i>完成后删除代理</i></div>"

        # 最终显示所有内容
        display(HTML(html_output))

        # 显示任何保存的图像
        for img_file in saved_images:
            display(Image(img_file))

# 执行函数
await run_agent_with_visualization()

AttributeError: 'AgentsClient' object has no attribute 'list_messages'